In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)
from scipy.stats import rankdata
from sklearn.metrics import roc_auc_score
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import spearmanr, kendalltau, rankdata, entropy
from scipy.spatial.distance import pdist, squareform
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
from sklearn.cluster import KMeans, DBSCAN
import joblib
import warnings
warnings.filterwarnings('ignore')
# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/playground-series-s6e2/sample_submission.csv
/kaggle/input/playground-series-s6e2/train.csv
/kaggle/input/playground-series-s6e2/test.csv
/kaggle/input/pss6e2-artifacts-oofs/optuna_artifacts/metadata.json
/kaggle/input/pss6e2-artifacts-oofs/optuna_artifacts/study_final.pkl
/kaggle/input/pss6e2-artifacts-oofs/optuna_artifacts/all_oof_preds.npy
/kaggle/input/pss6e2-artifacts-oofs/optuna_artifacts/all_models_info.csv
/kaggle/input/pss6e2-artifacts-oofs/optuna_artifacts/all_test_preds.npy
/kaggle/input/pss6e2-artifacts-oofs/optuna_artifacts/trial_oof_preds.pkl
/kaggle/input/pss6e2-artifacts-oofs/optuna_artifacts/trial_predictions.pkl
/kaggle/input/pss6e2-artifacts-oofs2/tabm/skewed_distributions.png
/kaggle/input/pss6e2-artifacts-oofs2/tabm/oof_TabM_D.csv
/kaggle/input/pss6e2-artifacts-oofs2/tabm/submission_TabM_D.csv
/kaggle/input/pss6e2-artifacts-oofs2/tabm/__results___files/__results___12_1.png
/kaggle/input/pss6e2-artifacts-oofs2/tabm/__results___files/__results___17_0.pn

In [2]:
class config:
    SEED = 42
    N_FOLDS = 5
    TARGET = 'Heart Disease'
    
    INPUT_DIR = '/kaggle/input/playground-series-s6e2'

class_mapping = {
    'Presence': 1,
    'Absence': 0
}
rev_class_mapping = {
    0: 'Absence',
    1: 'Presence'
}

CONFIG = config()

In [3]:
train = pd.read_csv('/kaggle/input/playground-series-s6e2/train.csv')
y = train['Heart Disease']
y_mapped = y.map({'Presence': 1, 'Absence': 0})
test = pd.read_csv('/kaggle/input/playground-series-s6e2/test.csv')
sample_submission = pd.read_csv('/kaggle/input/playground-series-s6e2/sample_submission.csv')

In [4]:
oofs_gblinear = np.load('/kaggle/input/pss6e2-artifacts-oofs2/gblinear/optuna_artifacts/all_oof_preds.npy')
test_gblinear = np.load('/kaggle/input/pss6e2-artifacts-oofs2/gblinear/optuna_artifacts/all_test_preds.npy')

oofs_gbtree = np.load('/kaggle/input/pss6e2-artifacts-oofs/optuna_artifacts/all_oof_preds.npy')
test_gbtree = np.load('/kaggle/input/pss6e2-artifacts-oofs/optuna_artifacts/all_test_preds.npy')

print(f"gblinear shapes: OOF={oofs_gblinear.shape}, Test={test_gblinear.shape}")
print(f"gbtree shapes: OOF={oofs_gbtree.shape}, Test={test_gbtree.shape}")


oofs_gbtree_df = pd.DataFrame(oofs_gbtree.T)
test_gbtree_df = pd.DataFrame(test_gbtree.T)

oofs_gblinear_df = pd.DataFrame(oofs_gblinear.T)  
test_gblinear_df = pd.DataFrame(test_gblinear.T)

oofs_gblinear_df.columns = [f'gblinear_model_{i}' for i in range(oofs_gblinear_df.shape[1])]
test_gblinear_df.columns = [f'gblinear_model_{i}' for i in range(test_gblinear_df.shape[1])]

oofs_gbtree_df.columns = [f'gbtree_model_{i}' for i in range(oofs_gbtree_df.shape[1])]
test_gbtree_df.columns = [f'gbtree_model_{i}' for i in range(test_gbtree_df.shape[1])]

realmlp_oof = pd.read_csv('/kaggle/input/pss6e2-artifacts-oofs2/realmlp_0.95398/oof.csv').drop(columns='id')
realmlp_test = pd.read_csv('/kaggle/input/pss6e2-artifacts-oofs2/realmlp_0.95398/submission.csv').drop(columns='id')

resnet50_oof = pd.read_csv('/kaggle/input/pss6e2-artifacts-oofs2/resnet50/oof_resnet50.csv').drop(columns='id')
resnet50_test = pd.read_csv('/kaggle/input/pss6e2-artifacts-oofs2/resnet50/submission_resnet50.csv').drop(columns='id')

tabm_oof = pd.read_csv('/kaggle/input/pss6e2-artifacts-oofs2/tabm/oof_TabM_D.csv').drop(columns='id')
tabm_test = pd.read_csv('/kaggle/input/pss6e2-artifacts-oofs2/tabm/submission_TabM_D.csv').drop(columns='id')

cat_oof = pd.read_csv('/kaggle/input/pss6e2-artifacts-oofs2/catboost/oof_original_0.955705176736493.csv').drop(columns='Unnamed: 0').rename(columns={'Heart Disease': 'cat_target'})
cat_test = pd.read_csv('/kaggle/input/pss6e2-artifacts-oofs2/catboost/submission_original_0.955705176736493.csv').drop(columns='id').rename(columns={'Heart Disease': 'cat_target'})

gblinear shapes: OOF=(24, 630000), Test=(24, 270000)
gbtree shapes: OOF=(120, 630000), Test=(120, 270000)


In [5]:
oofs_df = pd.concat([realmlp_oof,  resnet50_oof, tabm_oof, oofs_gblinear_df, oofs_gbtree_df, cat_oof], axis=1)
test_df = pd.concat([realmlp_test,  resnet50_test, tabm_test, test_gblinear_df, test_gbtree_df, cat_test], axis=1)

# oofs_df['gblinear_oof'] = oofs_gblinear_df['gblinear_model_23'].values
# test_df['gblinear_test'] = test_gblinear_df['gblinear_model_23'].values

In [6]:
print(f"Combined OOFs shape: {oofs_df.shape}")
print(f"Combined Tests shape: {test_df.shape}")

# print(f"\nModel types distribution:")
# print(f"GBLinear models: {len([c for c in oofs_df.columns if 'gblinear' in c])}")
# print(f"GBTree models: {len([c for c in oofs_df.columns if 'gbtree' in c])}")
# print(f"Total models: {all_oofs_df.shape[1]}")

Combined OOFs shape: (630000, 148)
Combined Tests shape: (270000, 148)


In [7]:
from tqdm.auto import tqdm
import time
from numba import njit

# 1. Numba-Accelerated AUC (The Speed Engine)
@njit
def fast_auc_numba(y_true, y_prob):
    """
    JIT-compiled AUC calculation. 
    This runs at C++ speeds, bypassing Python's slow loops.
    """
    # Sort indices by probability
    order = np.argsort(y_prob)[::-1]
    y_true_sorted = y_true[order]
    
    n_true = np.sum(y_true_sorted)
    n_false = len(y_true_sorted) - n_true
    
    if n_true == 0 or n_false == 0:
        return 0.5
        
    tp = 0
    auc = 0
    # Single pass through the sorted array to calculate area
    for i in range(len(y_true_sorted)):
        if y_true_sorted[i] == 1:
            tp += 1
        else:
            auc += tp
            
    return auc / (n_true * n_false)

# 2. Optimized Hill Climber
def optimized_hill_climb_final(df_oofs, y_true, batch_size=30, 
                               patience=30, std_dev=0.01, max_steps=1000):
    
    print("""
        \033[91m
        █▄░█ █░█ █▀▄▀█ █▄▄ ▄▀█   █▀▀ █▄░█ █▀▀ █ █▄░█ █▀▀
        █░▀█ █▄█ █░▀░█ █▄█ █▀█   ██▄ █░▀█ █▄█ █ █░▀█ ██▄
        \033[0m
        \033[96m>> Numba JIT Enabled | 150 Models | 600k Rows | No Subsampling\033[0m
        """)

    # Ensure float32 for faster matrix multiplication
    X = df_oofs.values.astype(np.float32)
    
    # Robust target conversion
    if hasattr(y_true, 'map'):
        y = y_true.values.astype(np.int32)
    else:
        y = y_true.astype(np.int32)
    
    n_models = X.shape[1]
    
    # Initialize weights
    best_weights = np.ones(n_models) / n_models
    # First prediction
    current_preds = X @ best_weights
    best_score = fast_auc_numba(y, current_preds)
    
    print(f"[*] Initial Baseline ROC-AUC: {best_score:.6f}")
    
    bad_steps = 0
    start_time = time.time()
    pbar = tqdm(range(max_steps), desc="Optimizing Weights")
    
    for step in pbar:
        print(f'current step: {step}')
        # Generate BATCH of weights at once
        # Nudge only a subset of models (20%) to find a better direction
        deltas = np.random.normal(0, std_dev, size=(batch_size, n_models))
        mask = np.random.rand(batch_size, n_models) > 0.2
        deltas[mask] = 0
        
        trial_weights = np.maximum(0, best_weights + deltas)
        trial_weights /= (trial_weights.sum(axis=1)[:, None] + 1e-12)
        
        # Matrix-Matrix Multiplication (Batch Prediction)
        # This calculates predictions for all 'batch_size' trials in one go
        all_preds = X @ trial_weights.T 
        
        found_better = False
        step_best_score = best_score
        step_best_weights = best_weights
        
        # Use the Numba function to check each trial in the batch
        for i in range(batch_size):
            score = fast_auc_numba(y, all_preds[:, i])
            if score > step_best_score:
                step_best_score = score
                step_best_weights = trial_weights[i].copy()
                found_better = True
        
        if found_better:
            improvement = step_best_score - best_score
            best_score = step_best_score
            best_weights = step_best_weights
            bad_steps = 0
        else:
            improvement = 0
            bad_steps += 1
            
        pbar.set_postfix({
            "AUC": f"{best_score:.6f}", 
            "Improv": f"{improvement:.2e}", 
            "P": f"{bad_steps}/{patience}"
        })
        
        if bad_steps >= patience:
            print(f"\n[!] Early Stopping triggered at step {step}")
            break
            
    print(f"\n[+] Optimization Finished in {time.time() - start_time:.2f}s")
    print(f"[+] Final AUC: {best_score:.6f}")
    
    return best_weights

# --- RUNNING THE CELL ---
# Ensure y_mapped is 0s and 1s
# best_weights = optimized_hill_climb_final(oofs_df, y_mapped)

In [8]:
%%time
best_weights = optimized_hill_climb_final(
    df_oofs=oofs_df, 
    y_true=y_mapped, 
     batch_size=30, 
    patience=400, std_dev=0.01, max_steps=20000
)


        
        █▄░█ █░█ █▀▄▀█ █▄▄ ▄▀█   █▀▀ █▄░█ █▀▀ █ █▄░█ █▀▀
        █░▀█ █▄█ █░▀░█ █▄█ █▀█   ██▄ █░▀█ █▄█ █ █░▀█ ██▄
        
        >> Numba JIT Enabled | 150 Models | 600k Rows | No Subsampling
        
[*] Initial Baseline ROC-AUC: 0.955524


Optimizing Weights:   0%|          | 0/20000 [00:00<?, ?it/s]

current step: 0
current step: 1
current step: 2
current step: 3
current step: 4
current step: 5
current step: 6
current step: 7
current step: 8
current step: 9
current step: 10
current step: 11
current step: 12
current step: 13
current step: 14
current step: 15
current step: 16
current step: 17
current step: 18
current step: 19
current step: 20
current step: 21
current step: 22
current step: 23
current step: 24
current step: 25
current step: 26
current step: 27
current step: 28
current step: 29
current step: 30
current step: 31
current step: 32
current step: 33
current step: 34
current step: 35
current step: 36
current step: 37
current step: 38
current step: 39
current step: 40
current step: 41
current step: 42
current step: 43
current step: 44
current step: 45
current step: 46
current step: 47
current step: 48
current step: 49
current step: 50
current step: 51
current step: 52
current step: 53
current step: 54
current step: 55
current step: 56
current step: 57
current step: 58
current

In [9]:
best_weights

array([2.52945908e-01, 3.62079503e-02, 1.01986955e-01, 4.37150949e-03,
       0.00000000e+00, 1.38703735e-03, 0.00000000e+00, 0.00000000e+00,
       0.00000000e+00, 0.00000000e+00, 0.00000000e+00, 2.19435964e-02,
       0.00000000e+00, 0.00000000e+00, 0.00000000e+00, 0.00000000e+00,
       0.00000000e+00, 0.00000000e+00, 0.00000000e+00, 0.00000000e+00,
       0.00000000e+00, 0.00000000e+00, 0.00000000e+00, 0.00000000e+00,
       0.00000000e+00, 0.00000000e+00, 0.00000000e+00, 0.00000000e+00,
       0.00000000e+00, 0.00000000e+00, 0.00000000e+00, 0.00000000e+00,
       1.13817424e-02, 0.00000000e+00, 6.61665685e-03, 0.00000000e+00,
       0.00000000e+00, 0.00000000e+00, 0.00000000e+00, 0.00000000e+00,
       0.00000000e+00, 0.00000000e+00, 1.97910442e-03, 3.94434128e-03,
       0.00000000e+00, 0.00000000e+00, 0.00000000e+00, 5.42762464e-04,
       0.00000000e+00, 0.00000000e+00, 0.00000000e+00, 5.29977243e-02,
       6.58882684e-03, 1.38741298e-03, 4.43485213e-02, 3.16138581e-02,
      

In [10]:
final_preds = test_df.values @ best_weights


In [11]:
# test_preds = test_gbtree_df['gbtree_model_59'].values
sample_submission[CONFIG.TARGET] = final_preds
sample_submission.to_csv('submission_csv_hillclimbing.csv', index=False)

In [12]:
sample_submission

,id,Heart Disease
0,630000,0.914769
1,630001,0.039613
2,630002,0.952508
3,630003,0.036127
4,630004,0.215465
...,...,...
269995,899995,0.169002
269996,899996,0.673665
269997,899997,0.075477
269998,899998,0.193661


In [13]:
# from sklearn.model_selection import StratifiedKFold
# from sklearn.linear_model import Ridge, LogisticRegression
# from sklearn.preprocessing import StandardScaler, LabelEncoder
# from tqdm import tqdm
# import numpy as np

# # Your data
# X_meta_train = all_oofs_df.values
# X_meta_test = all_tests_df.values
# y = train[CONFIG.TARGET].map(class_mapping).values

# strat_cols = ['Thallium', 'Chest pain type', 'Heart Disease']
# le = LabelEncoder()
# stratify_feature = le.fit_transform(train[strat_cols].astype(str).agg('_'.join, axis=1))

# print(f"Meta-train shape: {X_meta_train.shape}")
# print(f"Meta-test shape: {X_meta_test.shape}")

# # Stratified K-Fold setup
# skf = StratifiedKFold(n_splits=CONFIG.N_FOLDS, shuffle=True, random_state=CONFIG.SEED)

# # Storage for predictions
# meta_oof_preds = np.zeros(len(X_meta_train))
# meta_test_preds = np.zeros(len(X_meta_test))
# fold_scores = []

# print(f"\nTraining meta-model with {CONFIG.N_FOLDS}-fold CV")
# print("="*60)

# # CV loop
# for fold, (train_idx, val_idx) in enumerate(skf.split(X_meta_train, stratify_feature), 1):
    
#     print(f"\nFold {fold}:")
#     print(f"Train size: {len(train_idx)}, Val size: {len(val_idx)}")
    
#     # Split meta-data
#     X_train_fold = X_meta_train[train_idx]
#     X_val_fold = X_meta_train[val_idx]
#     y_train_fold = y[train_idx]
#     y_val_fold = y[val_idx]
    
#     # ===== FIXED: USE RIDGE, NOT LOGISTIC (for now) =====
#     # Scale features
#     scaler = StandardScaler()
#     X_train_scaled = scaler.fit_transform(X_train_fold)
#     X_val_scaled = scaler.transform(X_val_fold)
#     X_test_scaled = scaler.transform(X_meta_test)
    
#     # ===== OPTION A: RIDGE REGRESSION (WORKS) =====
#     ridge = Ridge(alpha=0.01, random_state=CONFIG.SEED + fold)
#     ridge.fit(X_train_scaled, y_train_fold)
    
#     # Predict
#     val_preds = ridge.predict(X_val_scaled)
#     test_preds = ridge.predict(X_test_scaled)
    
#     # ===== OPTION B: FIXED LOGISTIC REGRESSION =====
#     # Uncomment this if you want to try logistic
#     # logreg = LogisticRegression(
#     #     C=0.01,  # STRONG REGULARIZATION (1/alpha)
#     #     penalty='l2',
#     #     solver='lbfgs',  # BETTER FOR LARGE DATASETS
#     #     max_iter=1000,
#     #     random_state=CONFIG.SEED + fold
#     # )
#     # logreg.fit(X_train_scaled, y_train_fold)
#     # val_preds = logreg.predict_proba(X_val_scaled)[:, 1]
#     # test_preds = logreg.predict_proba(X_test_scaled)[:, 1]
    
#     # Clip predictions to [0, 1]
#     val_preds = np.clip(val_preds, 0, 1)
#     test_preds = np.clip(test_preds, 0, 1)
    
#     # Check for weird predictions
#     print(f"  Val preds range: [{val_preds.min():.4f}, {val_preds.max():.4f}]")
#     print(f"  Mean val pred: {val_preds.mean():.4f}")
    
#     # Store predictions
#     meta_oof_preds[val_idx] = val_preds
#     meta_test_preds += test_preds / CONFIG.N_FOLDS
    
#     # Score
#     fold_score = roc_auc_score(y_val_fold, val_preds)
#     fold_scores.append(fold_score)
    
#     print(f"  Fold {fold} AUC: {fold_score:.6f}")
#     print(f"  Ridge coef stats: mean={ridge.coef_.mean():.6f}, std={ridge.coef_.std():.6f}")

# # ===== FINAL RESULTS =====
# print(f"\n{'='*60}")
# print("META-MODEL CV RESULTS")
# print(f"{'='*60}")

# print(f"Fold scores: {[f'{s:.6f}' for s in fold_scores]}")
# print(f"Mean fold score: {np.mean(fold_scores):.6f} (±{np.std(fold_scores):.6f})")

# # Check for weird predictions in final OOF
# print(f"\nFinal OOF predictions analysis:")
# print(f"  Range: [{meta_oof_preds.min():.6f}, {meta_oof_preds.max():.6f}]")
# print(f"  Mean: {meta_oof_preds.mean():.6f}")
# print(f"  Std: {meta_oof_preds.std():.6f}")

# # Final score
# final_score = roc_auc_score(y, meta_oof_preds)
# print(f"\nOOF AUC Score: {final_score:.6f}")

# # Compare with simple average
# simple_avg_preds = np.mean(X_meta_train, axis=1)
# simple_avg_score = roc_auc_score(y, simple_avg_preds)
# print(f"Simple average baseline: {simple_avg_score:.6f}")
# print(f"Meta-model improvement: +{final_score - simple_avg_score:.6f}")

# # Create submission
# sample_submission['Heart Disease'] = meta_test_preds
# sample_submission.to_csv(f'submission_meta_ridge_{final_score:.6f}.csv', index=False)
# print(f"\n✅ Saved: submission_meta_ridge_{final_score:.6f}.csv")